**1. Import different config files**
We will start the notebook with OmegaConfig, which exists to combine multiple configuration files into one. This optimizes workflow and reduces the need for a crowded hierarchy of config files.

In [ ]:
from dataclasses import dataclass
from omegaconf import OmegaConf

market = OmegaConf.load("conf/market/aapl.yaml")
strategy = OmegaConf.load("conf/strategy/sma_crossover.yaml")
backtest = OmegaConf.load("conf/backtest/default.yaml")

cfg = OmegaConf.create({"market": market, "strategy": strategy, "backtest": backtest})
print(OmegaConf.to_yaml(cfg))
print(cfg.market.ticker, cfg.strategy.fast_window)

In [ ]:
dotlist = ["strategy.fast_window=5", "backtest.commission=0.002"]
override = OmegaConf.from_dotlist(dotlist)
cfg = OmegaConf.merge(cfg, override)
print(cfg.strategy.fast_window, cfg.backtest.commission)   # 5 0.002

In [ ]:
@dataclass
class StrategyConfig:
    name: str
    fast_window: int
    slow_window: int

schema = OmegaConf.structured(StrategyConfig)
checked = OmegaConf.merge(schema, strategy)                # passes
print(checked.slow_window)                                 # 30

# Break it on purpose, one line at a time (each raises, so run them separately):
OmegaConf.merge(schema, {"fast_window": "quick"})          # ValidationError: not an int
OmegaConf.merge(schema, {"fast_windw": 5})                 # ConfigKeyError: typo caught

**3. Hydra: groups and CLI overrides** \
\
Omegaconf has a built in configuration management tool called Hydra. it takes the config tree from a default list, gives it to the funtion, and lets the command long override anything. The folder names are the config groups (market, strategy, and backtest).\
\
If you look in the default.yaml, you will see the primary market, strategy, and backtest configs. Each line means "take conf/'group'/'option'.yaml and put it under the key 'group'". That is why segment 2 wrapped each file in market, strategy and backtest by hand.\
\
Now we head over to the run.py file, where using 'python run.py' in the terminal will output the results of the run using the primary configs.\
\
Output: RESULT AAPL sma 10/30: total_return=+26.6%, max_drawdown=-9.6% | final $12,658.

**3.1.  Override from the CLI**\
\
Run each line and read the output before the next:

In [ ]:
'''
python run.py strategy.fast_window=5 backtest.commission=0.002   # change values
python run.py market=tsla                                        # swap a whole group option
python run.py +backtest.note=trial1                              # + adds a key that does not exist
python run.py strategy.fast_window=50                            # trips the fast < slow check
python run.py --cfg job                                          # print the config, do not run main()
'''

**4: Multirun sweep**\
\
The -m flag runs run.py once per comma-separated value, with no change to the code.\
\
1: Launching 3 jobs locally, then three result lines identical to the ones in the Overview\
\
2: Sweep over the fast_window parameter with a value of 5, 10, and 20\
\
3: Sweep over two parameters, market and fast_window, with 2 different options each. This produces 4 different outcomes.

In [ ]:
#1) python run.py -m market=aapl,msft,tsla
#2) python run.py -m strategy.fast_window=5,10,20
#3) python run.py -m market=aapl,tsla strategy.fast_window=5,20

**5: Q&A, troubleshooting**\
\
...
